# NASA FIRMS Fire Detections: U.S., Canada, and Weather

This notebook downloads NASA FIRMS near-real-time detections for the United States and Canada, adds the weather nearest to each detection's acquisition hour, and displays the fires on an interactive basemap. The weather lookup uses the Open-Meteo Forecast API, which does **not** need an API key.

In [1]:
import importlib
import os
import sys
from typing import cast
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SOURCE_ROOT = PROJECT_ROOT / "src"
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root.")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import wildfire_data.weather_lookup as weather_lookup
importlib.reload(weather_lookup)
from wildfire_data.firms_collection import archive_firms_csv_response, record_firms_collection_failure
from wildfire_data.firms_date_ranges import firms_range_filename, next_firms_date_range, save_completed_firms_range
from wildfire_data.firms_preparation import prepare_firms_for_weather
from wildfire_data.weather_lookup import (
    fetch_weather_for_queries,
    prepare_weather_queries,
    sort_fires_for_export,
)

try:
    import folium
    from folium.plugins import HeatMap, MarkerCluster
except ImportError as exc:
    raise ImportError("Install the interactive-map dependency with `%pip install folium`, then rerun this cell.") from exc

try:
    from dotenv import load_dotenv
    load_dotenv(Path("config/.env"))
except ImportError:
    pass  # MAP_KEY can still be supplied as an environment variable.

MAP_KEY = (os.getenv("NASA_FIRMS_API_KEY") or os.getenv("MAP_KEY") or "").strip()
if not MAP_KEY:
    raise ValueError("Set NASA_FIRMS_API_KEY or MAP_KEY in your environment or .env file before running this notebook.")

print("Using NASA FIRMS map key from environment.")

Using NASA FIRMS map key from environment.


In [2]:
# This bbox covers the U.S. (including Alaska) and Canada.
# FIRMS bbox order: west,south,east,north.
# Each successful run records its range and the next run moves backward by this many inclusive days.
FIRMS_WINDOW_DAYS = 4
FIRMS_MIN_BRIGHT_TI4 = 305
FIRMS_RANGE_STATE_PATH = Path("data/state/firms_collection_range_state.json")
FIRMS_ARCHIVE_ROOT = Path("data")
FIRMS_RESULTS_DIRECTORY = Path("data/exports")
FIRMS_COLLECTION_REGION = "United States and Canada"
FIRMS_RESULTS_PREFIX = "fires_with_weather"
FIRMS_RANGE_STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
FIRMS_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
query_start_date, query_end_date = next_firms_date_range(
    FIRMS_RANGE_STATE_PATH,
    window_days=FIRMS_WINDOW_DAYS,
    results_directory=FIRMS_RESULTS_DIRECTORY,
    results_prefix=FIRMS_RESULTS_PREFIX,
)
bbox = "-179,24,-52,84"
product = "VIIRS_SNPP_NRT"

# Query one day at a time: the full U.S./Canada bounding box can time out for a multi-day VIIRS request.
retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))

daily_frames = []
daily_archives = []
for request_date in pd.date_range(query_start_date, query_end_date, freq="D"):
    date_string = request_date.date().isoformat()
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{product}/{bbox}/1/{date_string}"
    try:
        response = session.get(url, timeout=(10, 120))
    except requests.RequestException as exc:
        record_firms_collection_failure(
            FIRMS_ARCHIVE_ROOT,
            product=product,
            coverage_date=request_date.date(),
            region=FIRMS_COLLECTION_REGION,
            error=str(exc),
            retrieved_at=datetime.now(timezone.utc),
        )
        raise
    archive_result = archive_firms_csv_response(
        FIRMS_ARCHIVE_ROOT,
        payload=response.content,
        product=product,
        coverage_date=request_date.date(),
        region=FIRMS_COLLECTION_REGION,
        source_url=url,
        response_status_code=response.status_code,
        response_headers=dict(response.headers),
        request_parameters={"bbox": bbox, "days": 1},
        retrieved_at=datetime.now(timezone.utc),
        minimum_bright_ti4=FIRMS_MIN_BRIGHT_TI4,
    )
    daily_archives.append(archive_result)
    response.raise_for_status()
    daily_frames.append(pd.read_csv(StringIO(response.text)))

df = pd.concat(daily_frames, ignore_index=True)
session.close()
print(f"Fetched {len(df):,} FIRMS detections from {query_start_date} through {query_end_date} UTC.")
print(
    f"Archived {len(daily_archives):,} raw FIRMS responses and "
    f"{sum(len(result.normalized_artifacts) for result in daily_archives):,} normalized date partitions under {FIRMS_ARCHIVE_ROOT}."
)
df.head()

Fetched 3,824 FIRMS detections from 2026-05-23 through 2026-05-26 UTC.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,45.64240,-73.52372,298.82,0.40,0.45,2026-05-23,633,N,VIIRS,n,2.0NRT,283.86,1.11,N
1,45.72261,-63.67674,317.88,0.40,0.37,2026-05-23,633,N,VIIRS,n,2.0NRT,272.81,1.34,N
2,45.83530,-73.25233,313.85,0.39,0.44,2026-05-23,633,N,VIIRS,n,2.0NRT,278.88,1.43,N
3,45.83938,-73.25157,306.31,0.39,0.44,2026-05-23,633,N,VIIRS,n,2.0NRT,280.39,1.43,N
4,46.04288,-73.14574,305.97,0.39,0.44,2026-05-23,633,N,VIIRS,n,2.0NRT,282.40,0.87,N


In [3]:
# The collection archive retains every source field; this weather-view filter is non-destructive.
prepared_firms = prepare_firms_for_weather(
    df, minimum_bright_ti4=FIRMS_MIN_BRIGHT_TI4
)
plot_df = prepared_firms.fires

print(
    f"Prepared {len(plot_df):,} fire detections after removing "
    f"{prepared_firms.detections_with_required_fields - len(plot_df):,} below TI4 brightness "
    f"{FIRMS_MIN_BRIGHT_TI4}; retained all other FIRMS source fields."
)
plot_df.head()

Prepared 2,626 fire detections after removing 1,198 with TI4 brightness above 305.


,latitude,longitude,bright_ti4,acq_date,acq_time,acq_datetime,weather_lat,weather_lon,weather_hour
0,34.96644,-103.63808,367.0,2026-05-23,1938,2026-05-23 19:38:00+00:00,34.96644,-103.63808,2026-05-23 19:00:00+00:00
1,40.71920,-117.96160,367.0,2026-05-23,1940,2026-05-23 19:40:00+00:00,40.71920,-117.96160,2026-05-23 19:00:00+00:00
2,40.72534,-117.95968,367.0,2026-05-23,1940,2026-05-23 19:40:00+00:00,40.72534,-117.95968,2026-05-23 19:00:00+00:00
3,40.72871,-117.97276,367.0,2026-05-23,1940,2026-05-23 19:40:00+00:00,40.72871,-117.97276,2026-05-23 19:00:00+00:00
4,40.72921,-117.95524,367.0,2026-05-23,1940,2026-05-23 19:40:00+00:00,40.72921,-117.95524,2026-05-23 19:00:00+00:00


In [4]:
# Select deterministic input locations so every fire is within 1 km of a weather source.
# This cell only consolidates the requests; it does not call Open-Meteo.
WEATHER_BATCH_SIZE = 50
WEATHER_MAX_DISTANCE_M = 2_000
WEATHER_API_CALLS_PER_MINUTE = 600
WEATHER_CACHE_PATH = Path("data/weather/open_meteo_weather_cache.csv")
WEATHER_REQUESTS_PATH = Path("data/weather/open_meteo_weather_requests.csv")
WEATHER_RESULTS_DIRECTORY = Path("data/exports")
WEATHER_RESULTS_PREFIX = "fires_with_weather"
WEATHER_RATE_LIMIT_COOLDOWN_SECONDS = 90
WEATHER_MAX_CONSECUTIVE_RATE_LIMITS = 2

WEATHER_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
WEATHER_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

consolidated_sources, mapped_fires, weather_queries, weather_stats = prepare_weather_queries(
    plot_df, max_distance_m=WEATHER_MAX_DISTANCE_M
)
# This overwrites the previous request manifest for the new FIRMS range.
weather_queries.to_csv(WEATHER_REQUESTS_PATH, index=False)
print(
    f"Selected {weather_stats['weather_sources']:,} weather sources from "
    f"{weather_stats['input_locations']:,} unique fire locations."
)
print(
    f"Mapped {len(mapped_fires):,} original fire detections to those sources; "
    f"{weather_stats['source_hours']:,} source/hour lookups span "
    f"{weather_stats['source_days']:,} source days."
)
print(f"Saved {len(weather_queries):,} weather lookups to {WEATHER_REQUESTS_PATH}.")
print(consolidated_sources.to_string(index=False))
consolidated_sources

Selected 836 weather sources from 2,626 unique fire locations.
Mapped 2,626 original fire detections to those sources; 1,289 source/hour lookups span 1,055 source days.
Saved 1,289 weather lookups to open_meteo_weather_requests.csv.
 weather_source_lat  weather_source_lon weather_source_id                         weather_source_key
           66.44064          -159.27943     source_000000 0x1.09c33721d53cep+6:-0x1.3e8f1172ef0aep+7
           54.01854          -128.67899     source_000001 0x1.b025f84cad57cp+5:-0x1.015ba493c89f4p+7
           41.08303          -123.68852     source_000002 0x1.48aa0ba1f4b1fp+5:-0x1.eec10b630a915p+6
           43.11032          -123.55316     source_000003 0x1.58e1ef73c0c20p+5:-0x1.ee366f9335d25p+6
           39.69580          -123.15829     source_000004 0x1.3d90ff9724745p+5:-0x1.eca216c61522ap+6
           46.98843          -123.10944     source_000005 0x1.77e84dfce3151p+5:-0x1.ec70110a137f4p+6
           49.09650          -123.01852     source_000006 0x

,weather_source_lat,weather_source_lon,weather_source_id,weather_source_key
0,66.44064,-159.27943,source_000000,0x1.09c33721d53cep+6:-0x1.3e8f1172ef0aep+7
1,54.01854,-128.67899,source_000001,0x1.b025f84cad57cp+5:-0x1.015ba493c89f4p+7
2,41.08303,-123.68852,source_000002,0x1.48aa0ba1f4b1fp+5:-0x1.eec10b630a915p+6
3,43.11032,-123.55316,source_000003,0x1.58e1ef73c0c20p+5:-0x1.ee366f9335d25p+6
4,39.69580,-123.15829,source_000004,0x1.3d90ff9724745p+5:-0x1.eca216c61522ap+6
...,...,...,...,...
831,49.39899,-68.03675,source_000831,0x1.8b3121ab4b72cp+5:-0x1.1025a1cac0831p+6
832,45.69535,-64.73621,source_000832,0x1.6d9013a92a305p+5:-0x1.02f1e108c3f3ep+6
833,46.40223,-64.64133,source_000833,0x1.7337c45cbbc2cp+5:-0x1.0290b8cfbfc65p+6
834,45.72261,-63.67674,source_000834,0x1.6dc7e7c06e19cp+5:-0x1.fd69f6a93f291p+5


In [5]:
# Run this separately after reviewing weather_queries above.
fires_with_weather, weather_fetch_stats = fetch_weather_for_queries(
    mapped_fires,
    weather_queries,
    cache_path=WEATHER_CACHE_PATH,
    batch_size=WEATHER_BATCH_SIZE,
    requests_per_minute=WEATHER_API_CALLS_PER_MINUTE,
    rate_limit_cooldown_seconds=WEATHER_RATE_LIMIT_COOLDOWN_SECONDS,
    max_consecutive_rate_limits=WEATHER_MAX_CONSECUTIVE_RATE_LIMITS,
)
print(
    f"Weather calls: {weather_fetch_stats['projected_api_calls']:,} source/day calls planned "
    f"in {weather_fetch_stats['projected_requests']:,} HTTP batches "
    f"({weather_fetch_stats['projected_api_calls_without_cache']:,} calls without cache; "
    f"{weather_fetch_stats['cache_entries_retained']:,} current-range cache rows retained; "
    f"{weather_fetch_stats['cache_hits']:,} cached hourly lookups; "
    f"{weather_fetch_stats['http_attempts']:,} HTTP attempts; "
    f"{weather_fetch_stats['rate_limit_retries']:,} rate-limit retries)."
)
print(
    f"Added weather to {fires_with_weather['temperature_2m'].notna().sum():,} "
    f"of {len(fires_with_weather):,} detections."
)
if weather_fetch_stats["paused_for_rate_limit"]:
    print(
        f"Paused after two consecutive 429 responses with "
        f"{weather_fetch_stats['remaining_api_calls']:,} source/day calls remaining. "
        "Completed batches are saved; run the weather-resume cell to continue."
    )
else:
    fires_with_weather = sort_fires_for_export(fires_with_weather)
    WEATHER_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    WEATHER_RESULTS_PATH = WEATHER_RESULTS_DIRECTORY / firms_range_filename(
        query_start_date, query_end_date, prefix=WEATHER_RESULTS_PREFIX
    )
    fires_with_weather.to_csv(WEATHER_RESULTS_PATH, index=False)
    save_completed_firms_range(
        FIRMS_RANGE_STATE_PATH, query_start_date, query_end_date
    )
    print(f"Saved date-then-proximity ordered detections to {WEATHER_RESULTS_PATH}.")
    print(f"Recorded {query_start_date} through {query_end_date} as the completed FIRMS range.")
fires_with_weather.head()

Weather calls: 0 source/day calls planned in 0 HTTP batches (1,055 calls without cache; 1,289 current-range cache rows retained; 1,289 cached hourly lookups; 0 HTTP attempts; 0 rate-limit retries).
Added weather to 0 of 2,626 detections.
Saved date-then-proximity ordered detections to fires_with_weather_2026-05-23_to_2026-05-26.csv.
Recorded 2026-05-23 through 2026-05-26 as the completed FIRMS range.


,latitude,longitude,bright_ti4,acq_date,acq_time,acq_datetime,weather_lat,weather_lon,weather_hour,weather_source_id,weather_source_key,weather_source_lat,weather_source_lon,weather_source_distance_km,weather_observed_at,temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m
0,45.72261,-63.67674,317.88,2026-05-23,0633,2026-05-23 06:33:00+00:00,45.72261,-63.67674,2026-05-23 06:00:00+00:00,source_000834,0x1.6dc7e7c06e19cp+5:-0x1.fd69f6a93f291p+5,45.72261,-63.67674,0.000000,2026-05-23 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,46.04288,-73.14574,305.97,2026-05-23,0633,2026-05-23 06:33:00+00:00,46.04288,-73.14574,2026-05-23 06:00:00+00:00,source_000830,0x1.7057d1782d384p+5:-0x1.24953cddd6e05p+6,46.04288,-73.14574,0.000000,2026-05-23 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
2,45.83530,-73.25233,313.85,2026-05-23,0633,2026-05-23 06:33:00+00:00,45.83530,-73.25233,2026-05-23 06:00:00+00:00,source_000829,0x1.6eb290abb44e5p+5:-0x1.250297396d091p+6,45.83719,-73.25253,0.210644,2026-05-23 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
3,45.83938,-73.25157,306.31,2026-05-23,0633,2026-05-23 06:33:00+00:00,45.83938,-73.25157,2026-05-23 06:00:00+00:00,source_000829,0x1.6eb290abb44e5p+5:-0x1.250297396d091p+6,45.83719,-73.25253,0.254584,2026-05-23 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
4,27.20935,-80.73674,332.91,2026-05-23,0640,2026-05-23 06:40:00+00:00,27.20935,-80.73674,2026-05-23 06:00:00+00:00,source_000783,0x1.b3597f62b6ae8p+4:-0x1.42f26bf8769ecp+6,27.20935,-80.73674,0.000000,2026-05-23 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN


In [10]:
# Resume a weather fetch paused after consecutive 429 responses.
# The cache retains completed batches, so this requests only the remaining source/day lookups.
fires_with_weather, weather_fetch_stats = fetch_weather_for_queries(
    mapped_fires,
    weather_queries,
    cache_path=WEATHER_CACHE_PATH,
    batch_size=WEATHER_BATCH_SIZE,
    requests_per_minute=WEATHER_API_CALLS_PER_MINUTE,
    rate_limit_cooldown_seconds=WEATHER_RATE_LIMIT_COOLDOWN_SECONDS,
    max_consecutive_rate_limits=WEATHER_MAX_CONSECUTIVE_RATE_LIMITS,
)
if weather_fetch_stats["paused_for_rate_limit"]:
    print(
        f"Still paused; {weather_fetch_stats['remaining_api_calls']:,} source/day calls remain. "
        "Run this cell again later to resume."
    )
else:
    fires_with_weather = sort_fires_for_export(fires_with_weather)
    WEATHER_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    WEATHER_RESULTS_PATH = WEATHER_RESULTS_DIRECTORY / firms_range_filename(
        query_start_date, query_end_date, prefix=WEATHER_RESULTS_PREFIX
    )
    fires_with_weather.to_csv(WEATHER_RESULTS_PATH, index=False)
    save_completed_firms_range(
        FIRMS_RANGE_STATE_PATH, query_start_date, query_end_date
    )
    print(f"Saved date-then-proximity ordered detections to {WEATHER_RESULTS_PATH}.")
    print(f"Recorded {query_start_date} through {query_end_date} as the completed FIRMS range.")
fires_with_weather.head()

Saved date-then-proximity ordered detections to fires_with_weather_2026-05-27_to_2026-05-30.csv.
Recorded 2026-05-27 through 2026-05-30 as the completed FIRMS range.


,latitude,longitude,bright_ti4,acq_date,acq_time,acq_datetime,weather_lat,weather_lon,weather_hour,weather_source_id,weather_source_key,weather_source_lat,weather_source_lon,weather_source_distance_km,weather_observed_at,temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m
0,41.67309,-87.46759,314.90,2026-05-27,0659,2026-05-27 06:59:00+00:00,41.67309,-87.46759,2026-05-27 06:00:00+00:00,source_000959,0x1.4d627d028a1e0p+5:-0x1.5ddecfe9b7bf2p+6,41.67309,-87.46759,0.000000,2026-05-27 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,43.28014,-79.82716,313.75,2026-05-27,0659,2026-05-27 06:59:00+00:00,43.28014,-79.82716,2026-05-27 06:00:00+00:00,source_001122,0x1.5a3dba0a52696p+5:-0x1.3f4f0307f23cdp+6,43.28014,-79.82716,0.000000,2026-05-27 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
2,43.31415,-80.54985,306.47,2026-05-27,0659,2026-05-27 06:59:00+00:00,43.31415,-80.54985,2026-05-27 06:00:00+00:00,source_001104,0x1.5a836113404eap+5:-0x1.42330be0ded29p+6,43.31415,-80.54985,0.000000,2026-05-27 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
3,41.64278,-87.40990,318.87,2026-05-27,0659,2026-05-27 06:59:00+00:00,41.64278,-87.40990,2026-05-27 06:00:00+00:00,source_000961,0x1.4d224894c447cp+5:-0x1.5da3ec02f2f98p+6,41.64174,-87.41008,0.116478,2026-05-27 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN
4,41.62599,-87.14521,308.25,2026-05-27,0659,2026-05-27 06:59:00+00:00,41.62599,-87.14521,2026-05-27 06:00:00+00:00,source_000966,0x1.4d037b4a2339cp+5:-0x1.5c95ca6ca03c5p+6,41.62670,-87.14628,0.119034,2026-05-27 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN


In [11]:
# Reorder an existing result file by date, then proximity, without weather API calls.
from pathlib import Path

weather_results_prefix = globals().get("WEATHER_RESULTS_PREFIX", "fires_with_weather")
weather_results_directory = Path(globals().get("WEATHER_RESULTS_DIRECTORY", "data/exports"))
weather_results_path = Path(globals().get("WEATHER_RESULTS_PATH", weather_results_directory))
if not weather_results_path.is_file():
    dated_result_paths = list(weather_results_directory.glob(f"{weather_results_prefix}_*_to_*.csv"))
    if not dated_result_paths:
        raise FileNotFoundError("No dated weather-results CSV was found.")
    weather_results_path = max(dated_result_paths, key=lambda path: path.stat().st_mtime)
WEATHER_RESULTS_PATH = weather_results_path
saved_fires_with_weather = pd.read_csv(weather_results_path, dtype={"acq_time": "string"})
saved_fires_with_weather = sort_fires_for_export(saved_fires_with_weather)
temporary_weather_results_path = weather_results_path.with_name(f".{weather_results_path.name}.tmp")
saved_fires_with_weather.to_csv(temporary_weather_results_path, index=False)
temporary_weather_results_path.replace(weather_results_path)

print(f"Sorted {len(saved_fires_with_weather):,} rows by oldest acquisition time, then proximity.")
saved_fires_with_weather[["weather_source_distance_km", "acq_datetime"]].head()

Sorted 5,258 rows by oldest acquisition time, then proximity.


,weather_source_distance_km,acq_datetime
0,0.000000,2026-05-27 06:59:00+00:00
1,0.000000,2026-05-27 06:59:00+00:00
2,0.000000,2026-05-27 06:59:00+00:00
3,0.116478,2026-05-27 06:59:00+00:00
4,0.119034,2026-05-27 06:59:00+00:00


In [ ]:
# Each marker popup includes the weather associated with that fire detection.
# Set this to an integer (for example 5000) if an unusually dense result makes the HTML map too large.
MAX_MAPPED_DETECTIONS = None
map_df = fires_with_weather if MAX_MAPPED_DETECTIONS is None else fires_with_weather.head(MAX_MAPPED_DETECTIONS)

fire_map = folium.Map(location=[54, -106], zoom_start=3, tiles="CartoDB positron", control_scale=True)
HeatMap(map_df[["latitude", "longitude"]].values.tolist(), radius=12, blur=18, min_opacity=0.25, name="Fire density").add_to(fire_map)
markers = MarkerCluster(name="Fire detections").add_to(fire_map)

for fire in map_df.itertuples():
    weather_time = cast(datetime, getattr(fire, "weather_observed_at", pd.NaT))
    weather_time_text = "Unavailable" if pd.isna(weather_time) else weather_time.strftime("%Y-%m-%d %H:%M UTC")
    detected_at = cast(datetime, fire.acq_datetime)
    popup_html = f"""
    <b>Fire detection</b><br>
    Detected: {detected_at.strftime('%Y-%m-%d %H:%M UTC')}<br>
    Brightness (TI4): {fire.bright_ti4:.1f}<br><br>
    <b>Nearest hourly weather</b><br>
    Observation: {weather_time_text}<br>
    Temperature: {fire.temperature_2m} °C<br>
    Humidity: {fire.relative_humidity_2m} %<br>
    Precipitation: {fire.precipitation} mm<br>
    Wind speed: {fire.wind_speed_10m} km/h<br>
    WMO weather code: {fire.weather_code}
    """
    folium.CircleMarker(
        location=[float(cast(float, fire.latitude)), float(cast(float, fire.longitude))], radius=4, color="#d73027", fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(markers)

folium.LayerControl().add_to(fire_map)
fire_map